In [1]:
# ============================================================================
# EXTRACTION D'EMBEDDINGS AVEC DINOV2 À PARTIR DE SPECTROGRAMMES
# ============================================================================
# Ce notebook extrait des embeddings 2D à partir de spectrogrammes mel
# en utilisant le modèle Vision Transformer DINOv2-Large (pré-entraîné)
# Adapté pour Google Colab

# 1. Connexion à Google Drive (pour accéder aux données)
from google.colab import drive
drive.mount('/content/drive')

# 2. Import des bibliothèques essentielles
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import pandas as pd
from tqdm import tqdm  # Barre de progression
import numpy as np
import csv
import zipfile
import shutil
from pathlib import Path

print(" Google Drive montage successful")

Mounted at /content/drive
✓ Google Drive montage successful


In [2]:
# ============================================================================
# 2. INITIALISATION DU DEVICE ET DU MODÈLE DINOV2
# ============================================================================

# Détection automatique du GPU (accélération matérielle)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")

# Chargement du modèle DINOv2 Large (pré-entraîné sur vision)
print("Chargement du modèle DINOv2-Large...")
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').to(device)

# Optimisation GPU : utilisation de demi-précision (float16) pour réduire la mémoire
if device.type == 'cuda':
    model = model.half()
    print(" Mode demi-précision (FP16) activé pour GPU")

# Mode d'évaluation (pas de dropout, pas d'update des batch norms)
model.eval()
print(" Modèle en mode évaluation")

Device utilisé : cuda
Chargement du modèle DINOv2-Large...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:08<00:00, 148MB/s] 


✓ Mode demi-précision (FP16) activé pour GPU
✓ Modèle en mode évaluation


# ⚙️ Configuration pour Google Colab

## Préparation avant de lancer le notebook:

1. **Placez les données dans Google Drive:**
   - Créez un dossier `CNRS/Data/Audio/Spectrogramme/` dans votre Google Drive
   - Mettez-y le fichier `3s_mel_spectrograms.zip`
   - Créez un dossier `CNRS/embeddings/` pour les résultats

2. **Adaptez les chemins (Cellule 4):**
   - Modifiez `PATH_ZIP` avec le chemin correct vers votre ZIP
   - Modifiez `PATH_CSV_OUTPUT` pour le dossier des résultats

3. **Accélération GPU (recommandé):**
   - Allez dans Colab: Exécution → Modifier le type d'exécution → GPU

## Chemins typiques:
```
/content/drive/MyDrive/CNRS/Data/Audio/Spectrogramme/3s_mel_spectrograms.zip
/content/drive/MyDrive/CNRS/embeddings/dinov2_embeddings.csv
```

In [3]:
# ============================================================================
# 3. DÉFINITION DES TRANSFORMATIONS D'IMAGE POUR DINOV2
# ============================================================================

def get_dinov2_size(width, height=518):
    """
    Ajuste les dimensions de l'image aux contraintes de DINOv2.
    DINOv2 utilise des patches de 14x14, donc les dimensions doivent être
    multiples de 14 pour une performance optimale.

    Args:
        width: largeur de l'image originale
        height: hauteur cible (par défaut 518, qui est un multiple de 14)

    Returns:
        list: [hauteur, largeur_ajustée]
    """
    # La hauteur reste fixe (518 = 14*37)
    # La largeur est arrondie au multiple de 14 le plus proche
    new_w = int(round(width / 14) * 14)
    return [height, new_w]


# Pipeline de transformations appliquées à chaque image
transform = T.Compose([
    # Redimensionnement adaptatif à la largeur de l'image (hauteur: 518px)
    T.Lambda(lambda img: T.functional.resize(img, get_dinov2_size(img.size[0], img.size[1]))),
    # Conversion en tenseur (valeurs: 0-1)
    T.ToTensor(),
    # Normalisation ImageNet (statistiques pré-calculées sur millions d'images)
    # IMPORTANT: cette normalisation est critique pour DINOv2 (+14% accuracy)
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(" Pipeline de transformations défini")

✓ Pipeline de transformations défini


In [ ]:
# ============================================================================
# 4. CONFIGURATION DES CHEMINS ET EXTRACTION DU ZIP
# ============================================================================
# Pour Google Colab : adapter les chemins vers Google Drive
# Les données doivent être dans votre Google Drive

# === CONFIGURATION DES CHEMINS POUR COLAB ===
# IMPORTANT: Adapter ces chemins selon votre structure Google Drive
# Format: /content/drive/MyDrive/chemin/vers/vos/fichiers

# Chemin vers le ZIP depuis Google Drive
PATH_ZIP = "/content/drive/MyDrive/audio_simon_moutier/Data/Audio/Spectrogramme/3s_mel_spectrograms_subset.zip"

# Dossier temporaire local de Colab pour extraire les spectrogrammes
PATH_EXTRACTED = "/content/spectros_local/extracted"

# Chemin d'output du CSV dans Google Drive (pour persister les résultats)
PATH_CSV_OUTPUT = "/content/drive/MyDrive/audio_simon_moutier/embeddings/dinov2_embeddings_subset.csv"

print(f" Configuration des chemins pour Colab:")
print(f"   ZIP source (GDrive) : {PATH_ZIP}")
print(f"   Extraction locale : {PATH_EXTRACTED}")
print(f"   CSV output (GDrive) : {PATH_CSV_OUTPUT}")

# Vérification que le ZIP existe
if not os.path.exists(PATH_ZIP):
    raise FileNotFoundError(f"Le fichier ZIP n'existe pas : {PATH_ZIP}\n"
                           f"Vérifiez que:\n"
                           f"1. Le fichier est dans votre Google Drive au chemin correct\n"
                           f"2. Le chemin PATH_ZIP est correct (adapter si nécessaire)")

print(f" Fichier ZIP trouvé ({os.path.getsize(PATH_ZIP) / (1024**2):.1f} MB)")

# Nettoyage du dossier d'extraction s'il existe déjà
if os.path.exists(PATH_EXTRACTED):
    print(f"️  Suppression du dossier existant : {PATH_EXTRACTED}")
    shutil.rmtree(PATH_EXTRACTED)

# Extraction du ZIP
os.makedirs(PATH_EXTRACTED, exist_ok=True)
print(f" Extraction du ZIP en cours...")

with zipfile.ZipFile(PATH_ZIP, 'r') as zip_ref:
    zip_ref.extractall(PATH_EXTRACTED)

print(f" Extraction terminée")

# Détection du dossier réel des images (au cas où le ZIP contient un sous-dossier)
extracted_contents = os.listdir(PATH_EXTRACTED)
target_dir = PATH_EXTRACTED

# Si un seul sous-dossier a été créé, on l'utilise
if len(extracted_contents) == 1 and os.path.isdir(os.path.join(PATH_EXTRACTED, extracted_contents[0])):
    target_dir = os.path.join(PATH_EXTRACTED, extracted_contents[0])
    print(f"ℹSous-dossier détecté : {extracted_contents[0]}")

# Scan rapide des fichiers images
image_files = [f for f in os.listdir(target_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f" {len(image_files)} spectrogrammes trouvés")

📁 Configuration des chemins pour Colab:
   ZIP source (GDrive) : /content/drive/MyDrive/audio_simon_moutier/Data/Audio/Spectrogramme/3s_mel_spectrograms_subset.zip
   Extraction locale : /content/spectros_local/extracted
   CSV output (GDrive) : /content/drive/MyDrive/audio_simon_moutier/embeddings/dinov2_embeddings_subset.csv
✓ Fichier ZIP trouvé (1915.6 MB)
📦 Extraction du ZIP en cours...
✓ Extraction terminée
ℹ️  Sous-dossier détecté : 3s_mel_spectrograms_subset
✓ 10325 spectrogrammes trouvés


In [5]:
# ============================================================================
# 5. NETTOYAGE DES DIMENSIONS (DÉTECTION ANOMALIES)
# ============================================================================
# Cette étape vérifie que toutes les images ont des dimensions cohérentes.
# Les images de tailles aberrantes sont supprimées pour éviter les bugs
# dans DINOv2 lors du traitement par batch.

from collections import Counter

print("Scan des dimensions des images...")

# Récupération de la liste des fichiers images
all_files = [f for f in os.listdir(target_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
widths = []
heights = []

# Scan complet des dimensions
for f in tqdm(all_files, desc="Scan des dimensions"):
    try:
        with Image.open(os.path.join(target_dir, f)) as img:
            widths.append(img.size[0])
            heights.append(img.size[1])
    except Exception as e:
        print(f"Erreur en lisant {f}: {e}")

if not widths:
    raise ValueError("Aucune image valide trouvée!")

# Identification des dimensions majoritaires
width_counts = Counter(widths)
height_counts = Counter(heights)

main_width = width_counts.most_common(1)[0][0]
main_height = height_counts.most_common(1)[0][0]
width_freq = width_counts.most_common(1)[0][1]
height_freq = height_counts.most_common(1)[0][1]

print(f"\nStatistiques des dimensions:")
print(f"   Hauteur : {main_height}px (majoritaire: {height_freq}/{len(all_files)} images)")
print(f"   Largeur : {main_width}px (majoritaire: {width_freq}/{len(all_files)} images)")

# Suppression des images anormales
deleted_count = 0
for f in all_files:
    path = os.path.join(target_dir, f)
    try:
        with Image.open(path) as img:
            if img.size[0] != main_width or img.size[1] != main_height:
                os.remove(path)
                print(f"Supprimée : {f} (taille: {img.size[0]}x{img.size[1]})")
                deleted_count += 1
    except Exception as e:
        print(f"Erreur avec {f}: {e}")
        os.remove(path)
        deleted_count += 1

if deleted_count == 0:
    print("\nAucune anomalie détectée. Toutes les images sont cohérentes.")
else:
    print(f"\nNettoyage terminé : {deleted_count} image(s) supprimée(s).")

# Mise à jour de la liste des fichiers
image_files = [f for f in os.listdir(target_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"{len(image_files)} images prêtes pour l'extraction")

Scan des dimensions des images...


Scan des dimensions: 100%|██████████| 10325/10325 [00:03<00:00, 3002.66it/s]



Statistiques des dimensions:
   Hauteur : 518px (majoritaire: 10325/10325 images)
   Largeur : 522px (majoritaire: 10325/10325 images)

Aucune anomalie détectée. Toutes les images sont cohérentes.
10325 images prêtes pour l'extraction


In [ ]:
# ============================================================================
# 6. EXTRACTION DES EMBEDDINGS AVEC DINOV2
# ============================================================================

# Configuration du traitement par batch pour optimiser la vitesse
BATCH_SIZE = 128  # Nombre d'images traitées simultanément
NUM_WORKERS = 2  # Nombre de workers pour le chargement parallèle

print(f"\nConfiguration du traitement:")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Workers : {NUM_WORKERS}")
print(f"   Device : {device}")

# === CLASSE POUR CHARGER LES IMAGES ===
class SpectrogramDataset(Dataset):
    """
    Dataset personnalisé pour charger les spectrogrammes et les transformer.
    """
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.img_names = sorted([f for f in os.listdir(img_dir)
                                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.transform = transform

        if not self.img_names:
            raise ValueError(f"Aucune image trouvée dans {img_dir}")

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        # Chargement de l'image
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        image = Image.open(img_path).convert('RGB')

        # Application des transformations
        if self.transform:
            image = self.transform(image)

        return image, self.img_names[idx]


# === CRÉATION DU DATASET ET DATALOADER ===
print("\nCréation du dataset...")
dataset = SpectrogramDataset(target_dir, transform=transform)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Pas de mélange pour garder un ordre cohérent
    num_workers=NUM_WORKERS,
    pin_memory=True  # Accélération GPU
)

print(f"Dataset créé avec {len(dataset)} images")
print(f"{len(dataloader)} batches à traiter")

# === EXTRACTION DES EMBEDDINGS ===
print("\nExtraction des embeddings en cours...")

# Fichier de sortie
os.makedirs(os.path.dirname(PATH_CSV_OUTPUT), exist_ok=True)

# Listes d'accumulation
all_embeddings = []
all_filenames = []

# Inférence sur tous les batches
with torch.no_grad():  # Pas de calcul de gradients (économise mémoire)
    for batch_imgs, batch_names in tqdm(dataloader, desc="Extraction"):
        # Déplacement des images vers le device (CPU ou GPU)
        batch_imgs = batch_imgs.to(device)

        # Conversion en demi-précision si GPU
        if device.type == 'cuda':
            batch_imgs = batch_imgs.half()

        # Inférence : extraction des features du modèle
        # forward_features retourne un dictionnaire avec plusieurs clés
        with torch.cuda.amp.autocast() if device.type == 'cuda' else torch.no_grad():
            outputs = model.forward_features(batch_imgs)

        # Récupération du token CLS (classification token)
        # x_norm_clstoken : vecteur normalisé de taille [batch_size, 1024]
        embeddings = outputs["x_norm_clstoken"].cpu().float().numpy()

        # Accumulation des résultats
        all_embeddings.append(embeddings)
        all_filenames.extend(batch_names)

print(f"Extraction terminée")

# === SAUVEGARDE EN CSV ===
print(f"\nSauvegarde des embeddings en CSV...")

# Concaténation de tous les embeddings
all_embeddings = np.vstack(all_embeddings)

# Création d'un DataFrame pour manipulation facile
embedding_dim = all_embeddings.shape[1]
columns = ["filename"] + [f"dim_{i}" for i in range(embedding_dim)]

df = pd.DataFrame(
    np.column_stack([all_filenames, all_embeddings]),
    columns=columns
)

# Sauvegarde au format CSV
df.to_csv(PATH_CSV_OUTPUT, index=False)

print(f"Fichier sauvegardé : {PATH_CSV_OUTPUT}")
print(f"\nRésumé de l'extraction:")
print(f"Images traitées : {len(all_filenames)}")
print(f"Dimension des embeddings : {embedding_dim}")
print(f"Taille du fichier : {os.path.getsize(PATH_CSV_OUTPUT) / (1024**2):.2f} MB")

print("\nExtraction d'embeddings terminée avec succès!")


⚙️  Configuration du traitement:
   Batch size : 128
   Workers : 2
   Device : cuda

Création du dataset...
Dataset créé avec 10325 images
81 batches à traiter

Extraction des embeddings en cours...


Extraction:   0%|          | 0/81 [00:00<?, ?it/s]/tmp/ipykernel_2565/1842283322.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast() if device.type == 'cuda' else torch.no_grad():
Extraction: 100%|██████████| 81/81 [12:28<00:00,  9.24s/it]


Extraction terminée

Sauvegarde des embeddings en CSV...
Fichier sauvegardé : /content/drive/MyDrive/audio_simon_moutier/embeddings/dinov2_embeddings_subset.csv

Résumé de l'extraction:
Images traitées : 10325
Dimension des embeddings : 1024
Taille du fichier : 109.05 MB

Extraction d'embeddings terminée avec succès!


In [8]:
# ============================================================================
# 7. VALIDATION ET EXPLORATION DES RÉSULTATS (BONUS)
# ============================================================================
# Cette cellule vérifie la qualité des embeddings extraits et fournit
# des statistiques descriptives pour valider le processus.

print("Validation des résultats...\n")

# Chargement du CSV généré
df_results = pd.read_csv(PATH_CSV_OUTPUT)

print(f"Dimensions du CSV:")
print(f"Lignes : {len(df_results)}")
print(f"Colonnes : {len(df_results.columns)}")

# Extraction des valeurs d'embeddings (toutes les colonnes sauf 'filename')
embeddings_values = df_results.iloc[:, 1:].values

print(f"\nStatistiques des embeddings:")
print(f"Min : {embeddings_values.min():.4f}")
print(f"Max : {embeddings_values.max():.4f}")
print(f"Moyenne : {embeddings_values.mean():.4f}")
print(f"Écart-type : {embeddings_values.std():.4f}")

# Vérification de la normalisation (DINOv2 produit des embeddings normalisés)
norms = np.linalg.norm(embeddings_values, axis=1)
print(f"\nNorme des embeddings (devrait être ~1.0 après normalisation):")
print(f"Moyenne : {norms.mean():.4f}")
print(f"Min/Max : {norms.min():.4f} / {norms.max():.4f}")

# Affichage de quelques exemples
print(f"\nPremiers embeddings extraits:")
print(df_results.head())

Validation des résultats...

Dimensions du CSV:
Lignes : 10325
Colonnes : 1025

Statistiques des embeddings:
Min : -8.5907
Max : 6.6472
Moyenne : -0.0004
Écart-type : 1.4587

Norme des embeddings (devrait être ~1.0 après normalisation):
Moyenne : 46.6690
Min/Max : 40.2703 / 48.1307

Premiers embeddings extraits:
                                            filename     dim_0     dim_1  \
0  alarm_call_HiPsh435_audio_2022-03-06_23-59-48-...  2.392429  1.443002   
1  alarm_call_HiPsh435_audio_2022-03-07_00-59-53-...  1.579869  1.673332   
2  alarm_call_HiPsh435_audio_2022-03-07_00-59-53-...  0.862197  1.560927   
3  alarm_call_HiPsh435_audio_2022-03-07_00-59-53-...  1.279355  1.241777   
4  alarm_call_HiPsh435_audio_2022-03-07_00-59-53-...  2.662179  0.957081   

      dim_2     dim_3     dim_4     dim_5     dim_6     dim_7     dim_8  ...  \
0 -1.711983  1.361079 -3.405027 -0.529888  0.145898 -1.290463  1.183500  ...   
1 -0.873610  1.146980 -3.718072 -0.710223 -0.051819 -0.626691  0.6916

In [ ]:
# ============================================================================
# 8. FINALISATION ET SYNCHRONISATION GOOGLE DRIVE (COLAB ONLY)
# ============================================================================
# Cette cellule synchronise les données avec Google Drive et marque la fin
# du processus. À exécuter uniquement sur Colab.

import time

print(" Synchronisation des données vers Google Drive...")

try:
    # Forcer l'écriture en mémoire et synchroniser avec Google Drive
    from google.colab import drive
    drive.flush_and_unmount()
    print(" Synchronisation réussie - Drive démonté")
except Exception as e:
    print(f"️  Erreur lors de la synchronisation : {e}")
    print("   (Les données sont probablement déjà sauvegardées)")

# Pause de sécurité
time.sleep(2)

print("EXTRACTION COMPLÈTE!")